In [ ]:
import kagglehub
import os
import pandas as pd

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Build the full CSV path
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)
df.head()



In [ ]:

# 2. Inspect the first few rows
print("HEAD:")
display(df.head())


In [ ]:
# Task 3: Write your code here:
# 3. Display dataset information
print("\nINFO:")
df.info()

In [ ]:
# Task 4: Write your code here:
# 4. Statistical description
print("\nDESCRIBE:")
display(df.describe())

In [ ]:
# Task 1: Write your code here:
# Check how many missing values exist in each column df.isnull().sum()
df.isnull().sum()
df = df.fillna(df.median(numeric_only=True))

In [ ]:
# Task 2: Write your code here:
duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)
df = df.drop_duplicates()


In [ ]:
# Task 3: Write your code here:
# Identify categorical columns
cat_cols = df.select_dtypes(include=['object']).columns
print("Categorical columns:", cat_cols)

# Apply one-hot encoding if categorical columns exist
if len(cat_cols) > 0:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

# Separate features and target
X = df.drop(columns='Target')
y = df['Target']

# Apply StandardScaler to numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# Task 5: Write your code here:
print("Target distribution (percentage):")
print(y.value_counts(normalize=True))


In [ ]:
# Task 1: Write your code here:

X = df.drop(columns="Target")   # all features
y = df["Target"]                # target column


In [ ]:
# Task 2,3,4,5: Write your code here:
!pip install catboost

# Split features and target
X = df.drop(columns="Target")
y = df["Target"]

# Use StratifiedKFold
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
import numpy as np

f1_scores = []

for train_index, test_index in skf.split(X, y):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = CatBoostClassifier(verbose=0, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    f1_scores.append(f1)

print("Average F1 Score across folds:", np.mean(f1_scores))


In [ ]:
# Task 1: Write your code here:
# Plot Feature Importance

import matplotlib.pyplot as plt
import numpy as np

# Get feature importance values from the last trained model
importances = model.get_feature_importance()
feature_names = X.columns

# Sort features by importance
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.barh(feature_names[sorted_idx][:20], importances[sorted_idx][:20])
plt.gca().invert_yaxis()
plt.title("Top 20 Most Important Features (CatBoost)")
plt.xlabel("Importance Score")
plt.show()


In [ ]:
# Task 2: Write your code here:
# Identify the Golden Feature

# Find index of the most important feature
golden_idx = np.argmax(importances)

# Print the name of the golden feature
golden_feature = feature_names[golden_idx]
print("The Golden Feature is:", golden_feature)


In [ ]:
# Task Bonus: Write your code here:
# Create X using only the golden feature
X_golden = df[[golden_feature]]
y = df["Target"]

  # Use StratifiedKFold again
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

golden_f1_scores = []

# Cross-validation loop using only the golden feature
for train_index, test_index in skf.split(X_golden, y):

    X_train, X_test = X_golden.iloc[train_index], X_golden.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model_golden = CatBoostClassifier(verbose=0, random_state=42)
    model_golden.fit(X_train, y_train)

    y_pred = model_golden.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    golden_f1_scores.append(f1)

# Print comparison
print("Average F1 Score (Full Model):", np.mean(f1_scores))
print("Average F1 Score (Golden Feature Only):", np.mean(golden_f1_scores))
